## ETL Silver – Forecast (diario y horario)

### Propósito
Transformar los JSON raw de pronóstico (capa Bronze) en tablas Delta Silver:
- una tabla **diaria** (1 fila por `city + date`)
- una tabla **horaria** (1 fila por `city + time`)
Ambas quedan deduplicadas por `ingestion_time` (último forecast disponible).

### Entrada
- Ruta Bronze:
  - `/Volumes/workspace/default/bronce_clima/forecast/`
- Formato: JSON con payload en `data.forecast.forecastday` + `metadata.ingestion_time`

---

## 1) Tabla Silver diaria: `weather_daily_silver_forecast`

### Transformaciones principales
- Explosión del array:
  - `explode(data.forecast.forecastday)` → 1 fila por día pronosticado
- Selección de métricas desde `forecastday.day`:
  - `maxtemp_c`, `mintemp_c`, `avgtemp_c`, `avghumidity`, `totalprecip_mm`, `maxwind_kph`,
    `daily_chance_of_rain`, `uv`
- Conversión de `ingestion_time` a timestamp:
  - `to_timestamp(ingestion_time, "yyyy-MM-dd'T'HH-mm-ss'Z'")`
- Deduplicación (último forecast):
  - `row_number() over (partition by city, date order by ingestion_time desc)` y se conserva `row_num = 1`

### Salida
- Tabla Delta (managed): `weather_daily_silver_forecast`
- Granularidad: `city + date`

---

## 2) Tabla Silver horaria: `weather_hourly_silver_forecast`

### Transformaciones principales
- Explosión de horas:
  - `explode(day.hour)` → 1 fila por hora pronosticada
- Selección de métricas horarias:
  - `time`, `temp_c`, `feelslike_c`, `uv`, `precip_mm`, `humidity`, `wind_kph`, `cloud`,
    `chance_of_rain`, `will_it_rain`, `ingestion_time`
- Tipado de campos:
  - `time` → timestamp (`to_timestamp(time, "yyyy-MM-dd HH:mm")`)
  - `date` → derivada desde `time` (`to_date(time)`)
  - `hour` → extraída desde `time` (`hour(time)`)
  - `ingestion_time` → timestamp (`yyyy-MM-dd'T'HH-mm-ss'Z'`)
- Deduplicación (último forecast):
  - `row_number() over (partition by city, time order by ingestion_time desc)` y se conserva `row_num = 1`

### Salida
- Tabla Delta (managed): `weather_hourly_silver_forecast`
- Granularidad: `city + time`

---

### Notas
- La deduplicación asegura que para una misma `city` y `date/time` quede la predicción más reciente.
- `date` en la tabla horaria se deriva desde `time` para facilitar agregaciones diarias posteriores.

In [0]:
from pyspark.sql.functions import col,explode,lower, regexp_replace,desc,row_number,to_timestamp,to_date,hour
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
df_forecast_bronze = spark.read.json("/Volumes/workspace/default/bronce_clima/forecast/")
df_forecast_bronze.show(5)
df_forecast_bronze.printSchema()

In [0]:

df_days = df_forecast_bronze.select(
    col("metadata.ciudad").alias("city"),
    col("date"),
    col("metadata.ingestion_time").alias("ingestion_time"),
    explode(col("data.forecast.forecastday")).alias("day")
)

In [0]:
forecast_daily = df_days.select(
    col("city"),
    col("day.date").alias("date"),
    col("day.day.maxtemp_c").alias("maxtemp_c"),
    col("day.day.mintemp_c").alias("mintemp_c"),
    col("day.day.totalprecip_mm").alias("totalprecip_mm"),
    col("day.day.daily_chance_of_rain").alias("daily_chance_of_rain"),
    col("ingestion_time"),
    col("day.day.uv"),
    col("day.day.avgtemp_c").alias("avgtemp_c"),
    col("day.day.avghumidity").alias("avghumidity"),
    col("day.day.maxwind_kph").alias("maxwind_kph")
)

In [0]:
forecast_daily.printSchema(5)
forecast_daily.show(5)

In [0]:
forecast_daily = forecast_daily.withColumn(
    "ingestion_time",
    to_timestamp("ingestion_time", "yyyy-MM-dd'T'HH-mm-ss'Z'")
)

window_spec = Window.partitionBy("city", "date").orderBy(desc("ingestion_time"))

forecast_daily_latest = (
    forecast_daily
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

In [0]:
forecast_daily_latest.show()

In [0]:
forecast_daily_latest.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_daily_silver_forecast")

In [0]:
spark.table("weather_daily_silver_forecast").show(5)

In [0]:
df_hourly = (
    df_days
    .withColumn("hour", explode(col("day.hour")))
)

In [0]:
forecast_hourly = df_hourly.select(
    col("city"),
    col("date"),
    col("hour.time").alias("time"),
    col("hour.feelslike_c").alias("feelslike_c"),
    col("hour.uv").alias("uv"),
    col("hour.temp_c").alias("temp_c"),
    col("hour.precip_mm").alias("precip_mm"),
    col("hour.humidity").alias("humidity"),
    col("hour.wind_kph").alias("wind_kph"),
    col("hour.cloud").alias("cloud"),
    col("hour.chance_of_rain").alias("chance_of_rain"),
    col("hour.will_it_rain").alias("will_it_rain"),
    col("ingestion_time")
)


In [0]:
forecast_hourly.printSchema()
forecast_hourly.show(5)

In [0]:
#Convertir time
forecast_hourly = forecast_hourly.withColumn(
    "time",
    to_timestamp("time", "yyyy-MM-dd HH:mm")
)

# Arreglar date
forecast_hourly = forecast_hourly.withColumn(
    "date",
    to_date("time")
)

# Extraer hour
forecast_hourly = forecast_hourly.withColumn(
    "hour",
    hour("time")
)

#Convertir ingestion_time
forecast_hourly = forecast_hourly.withColumn(
    "ingestion_time",
    to_timestamp("ingestion_time", "yyyy-MM-dd'T'HH-mm-ss'Z'")
)

In [0]:
forecast_hourly.printSchema()
forecast_hourly.show(5)

In [0]:
window_spec = Window.partitionBy("city", "time").orderBy(desc("ingestion_time"))

forecast_hourly_latest = (
    forecast_hourly
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)


In [0]:
forecast_hourly_latest.show(5)

In [0]:
forecast_hourly_latest.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_hourly_silver_forecast")

In [0]:
spark.table("weather_hourly_silver_forecast").show(5)